# Arithmetic

EasyScience numbers know their **unit**, their **uncertainty** and, for a `Parameter`, the **bounds** a fit
may explore. The operators `+`, `-`, `*`, `/` and `**`, negation and `abs` work on them and on plain
numbers, and every result is a **new** object whose value, unit, variance and bounds are worked out for you.

This notebook shows how that works and how it is expected to behave. Each part closes with a few exercises on what was just shown. Their hints and solutions are folded away and open on a click, so the text reads straight through; a solution is a worked answer with its result.

For independent inputs the variance of a result is propagated to first order,

$$
\sigma_f^2 = \sum_i \left(\frac{\partial f}{\partial x_i}\right)^2 \sigma_i^2 .
$$

In [ ]:
import numpy as np
import plotly.graph_objects as go

import easyscience as es
from easyscience import DescriptorNumber
from easyscience import Parameter

The function `show` prints a number as value, error and unit, and, for a `Parameter` with finite bounds, the bounds.

In [ ]:
def show(number):
    """One line with value, error, unit and, for a Parameter, the finite bounds."""
    text = f'{number.value:.4f} ± {number.error or 0.0:.4f} {number.unit}'
    if hasattr(number, 'min') and (np.isfinite(number.min) or np.isfinite(number.max)):
        text += f'   in [{number.min:.4f}, {number.max:.4f}]'
    return text

In [ ]:
import base64
import contextlib
import html
import io
import re

from IPython import get_ipython
from IPython.display import HTML
from IPython.display import IFrame
from IPython.display import display
from pygments import highlight
from pygments.formatters import HtmlFormatter
from pygments.lexers import PythonLexer


def show_figure(fig: go.Figure, height: int = 420) -> None:
    """Show a Plotly figure as an embedded iframe, so that it renders in the docs."""
    page = (
        go
        .Figure(fig)
        .update_layout(template='plotly_white')
        .to_html(
            full_html=True,
            include_plotlyjs='cdn',
            default_width='100%',
            default_height=f'{height - 20}px',
        )
    )
    encoded = base64.b64encode(page.encode('utf-8')).decode('ascii')
    display(IFrame(src=f'data:text/html;base64,{encoded}', width='100%', height=height))


def solution(note: str, source: str) -> None:
    """Run a solution in a copy of the notebook namespace and show it, folded, with its output."""
    code = source.strip('\n')
    namespace = dict(get_ipython().user_ns)
    stream = io.StringIO()
    with contextlib.redirect_stdout(stream):
        exec(code, namespace)
    parts = [
        '<details><summary>Solution</summary>',
        highlight(code, PythonLexer(), HtmlFormatter()),
    ]
    result = stream.getvalue().rstrip('\n')
    if result:
        parts.append('<p>Output:</p><pre>' + html.escape(result) + '</pre>')
    if note.strip():
        text = re.sub('`([^`]+)`', r'<code>\1</code>', html.escape(note.strip()))
        parts.append('<p>' + text + '</p>')
    parts.append('</details>')
    display(HTML(''.join(parts)))


get_ipython().register_magic_function(solution, magic_kind='cell')

## Numbers with an uncertainty

A `Parameter` has a value, a unit, an uncertainty and bounds. Here is a length of 2 m with an uncertainty of
0.2 m, which a fit may vary between 0 and 10 m:

In [ ]:
length = Parameter(name='length', value=2.0, unit='m', variance=0.04, min=0.0, max=10.0)
length

The uncertainty is stored as a **variance**, and `error` is its square root:

In [ ]:
print(length.value, length.unit)
print(length.variance, length.error)
print(length.min, length.max)

A `DescriptorNumber` is the same without the fitting bounds. It is the right choice for a number that is
known but is never fitted, such as a measured constant:

In [ ]:
scale = DescriptorNumber(name='scale', value=3.0, variance=0.01)
print(show(length))
print(show(scale))

## Addition and subtraction

The operators work between `Parameter`s, `DescriptorNumber`s and plain numbers. The result is a `Parameter`
(with bounds) if any operand is a `Parameter`, and a `DescriptorNumber` otherwise.

Only numbers with the same physical dimension can be added. If the units differ, the second operand is
converted to the unit of the first, so the result is in metres here:

In [ ]:
width = Parameter(name='width', value=50.0, unit='cm', variance=4.0, min=10.0, max=100.0)
total = length + width
print(show(total))

The variances add, $0.2^2 + 0.02^2 = 0.0404\ \text{m}^2$, and so do the bounds: $[0 + 0.1,\ 10 + 1.0]$ m.
Subtraction combines the bounds the other way round: the lowest possible difference is the minimum of the
first minus the maximum of the second.

In [ ]:
difference = length - width
print(show(difference))

A plain number can only be added to a dimensionless value. Adding one to a length is an error, while a
dimensionless `DescriptorNumber` accepts it:

In [ ]:
try:
    length + 1.0
except Exception as error:
    print(f'{type(error).__name__}: {error}')

print(show(scale + 1.0))

### Exercise 1: Total length

A rod is built from two sections that were measured separately: 1.20 ± 0.02 m and 0.80 ± 0.03 m. What is the total length, with its uncertainty?

In [ ]:
section_1 = Parameter(name='section_1', value=1.20, variance=0.02**2, unit='m')
section_2 = Parameter(name='section_2', value=0.80, variance=0.03**2, unit='m')

<details>
<summary>Hint 1</summary>

Two lengths in the same unit can simply be added with `+`.

</details>

<details>
<summary>Hint 2</summary>

The variances add: $\sigma^2 = 0.02^2 + 0.03^2$, and `error` is the square root of the variance.

</details>

In [ ]:
%%solution
total = section_1 + section_2
print(show(total))

### Exercise 2: Mixed units

A tube is 1.20 ± 0.02 m long and a gap of 12.5 ± 0.5 cm has to be cut out of it. How long is the rest, in metres?

In [ ]:
tube = Parameter(name='tube', value=1.20, variance=0.02**2, unit='m')
gap = Parameter(name='gap', value=12.5, variance=0.5**2, unit='cm')

<details>
<summary>Hint 1</summary>

Subtract the gap from the tube; the units do not have to match.

</details>

<details>
<summary>Hint 2</summary>

The second operand is converted to the unit of the first, so the result is in the unit of `tube`.

</details>

In [ ]:
%%solution
remaining = tube - gap
print(show(remaining))

## Multiplication and division

Units multiply and divide. The area of a rectangle, with the sides in the same unit:

In [ ]:
height = Parameter(name='height', value=1.5, unit='m', variance=0.01, min=1.0, max=3.0)
area = length * height
print(show(area))

The bounds of a product are the lowest and highest of the four products of the bounds. A speed is a length
over a time:

In [ ]:
time = Parameter(name='time', value=4.0, unit='s', variance=0.0625, min=2.0, max=8.0)
speed = length / time
print(show(speed))

Multiplying a `DescriptorNumber` by a plain number gives a `DescriptorNumber`, so there are no bounds:

In [ ]:
doubled = scale * 2
print(type(doubled).__name__, show(doubled))

### Exercise 3: Speed

A cart covers 3.0 ± 0.1 m in 1.5 ± 0.05 s. What is its speed, with its uncertainty and unit?

In [ ]:
distance = Parameter(name='distance', value=3.0, variance=0.1**2, unit='m')
time = Parameter(name='time', value=1.5, variance=0.05**2, unit='s')

<details>
<summary>Hint 1</summary>

Speed is distance divided by time.

</details>

<details>
<summary>Hint 2</summary>

Units divide as well: look at the unit of the result.

</details>

In [ ]:
%%solution
speed = distance / time
print(show(speed))

## Powers, negation and absolute value

The exponent of a power is a plain number. Negation flips the bounds, and `abs` folds them:

In [ ]:
print(show(length**2))
print(show(-length))
print(show(abs(-length)))

### Exercise 4: Area of a beam spot

The radius of a circular beam spot is 2.0 ± 0.1 cm. What is its area?

In [ ]:
radius = Parameter(name='radius', value=2.0, variance=0.1**2, unit='cm')

<details>
<summary>Hint 1</summary>

The area is $\pi r^2$. `np.pi` is a plain number, and `**` takes a plain-number exponent.

</details>

<details>
<summary>Hint 2</summary>

A plain number can multiply a `Parameter`; the result is again a `Parameter`.

</details>

In [ ]:
%%solution The lower bound 0 appears because a square can never be negative.
area = np.pi * radius**2
print(show(area))

## Uncertainty in a chart

The uncertainty of a result depends on the inputs. A speed $v = d/t$ of an object that covered
$3.0 \pm 0.1$ m is much less certain for short times than for long ones, even though the time has the same
uncertainty of 0.05 s each time. The chart is interactive: hover over the curve to read the values. It loads
Plotly from the internet, so it needs a connection. The charts are shown with a small helper,
`show_figure(fig)`, which is defined in a hidden cell; in your own notebook `fig.show()` does the same.

In [ ]:
times = np.linspace(0.5, 5.0, 46)
distance = Parameter(name='distance', value=3.0, unit='m', variance=0.1**2)

values, errors = [], []
for value in times:
    time = Parameter(name='time', value=value, unit='s', variance=0.05**2)
    speed = distance / time
    values.append(speed.value)
    errors.append(speed.error)
values, errors = np.array(values), np.array(errors)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=times,
        y=values - errors,
        mode='lines',
        line=dict(width=0),
        showlegend=False,
        hoverinfo='skip',
    )
)
fig.add_trace(
    go.Scatter(
        x=times,
        y=values + errors,
        mode='lines',
        line=dict(width=0),
        fill='tonexty',
        fillcolor='rgba(99, 110, 250, 0.25)',
        name='± 1 error',
        hoverinfo='skip',
    )
)
fig.add_trace(
    go.Scatter(
        x=times,
        y=values,
        mode='lines',
        name='speed',
        line=dict(color='rgb(99, 110, 250)'),
        customdata=errors,
        hovertemplate='time = %{x:.2f} s<br>speed = %{y:.2f} ± %{customdata:.2f} m/s<extra></extra>',
    )
)
fig.update_layout(
    xaxis_title='time / s',
    yaxis_title='speed / (m/s)',
    margin=dict(l=60, r=20, t=30, b=50),
    legend=dict(x=0.6, y=0.95),
)
show_figure(fig)

Variances add, so the error of a **sum** of $n$ equal measurements grows like $\sigma\sqrt{n}$, while the
error of their **mean** shrinks like $\sigma/\sqrt{n}$. Here each measurement is $1.0 \pm 0.1$:

In [ ]:
counts = np.arange(1, 26)

sum_errors, mean_errors = [], []
for n in counts:
    total = Parameter(name='measurement', value=1.0, variance=0.1**2)
    for _ in range(int(n) - 1):
        total = total + Parameter(name='measurement', value=1.0, variance=0.1**2)
    sum_errors.append(total.error)
    mean_errors.append((total / int(n)).error)

fig = go.Figure()
fig.add_trace(go.Scatter(x=counts, y=sum_errors, mode='lines+markers', name='error of the sum'))
fig.add_trace(go.Scatter(x=counts, y=mean_errors, mode='lines+markers', name='error of the mean'))
fig.update_layout(
    xaxis_title='number of measurements n',
    yaxis_title='error',
    margin=dict(l=60, r=20, t=30, b=50),
    legend=dict(x=0.05, y=0.95),
)
show_figure(fig, height=380)

## Good to know

An operator knows nothing about where its operands come from. The same object on both sides of an operator
gives the right variance: $x - x$ is exactly $0 \pm 0$. But an intermediate result is a **new, independent
number**: in $(2x)/x$ the $x$ cancels and the exact answer is $2 \pm 0$, yet `2 * x` no longer remembers
that it came from `x`. Bounds are combined in the same way, as if the operands were independent, so
$x - x$ of a `Parameter` that may vary between 0 and 10 has bounds $[-10, 10]$:

In [ ]:
x = Parameter(name='x', value=10.0, variance=0.1**2, min=0.0, max=10.0)
print(show(x - x))
print(show((2 * x) / x))

The same happens with quantities that are correlated, such as the slope and the intercept of one fit: the
operators treat them as independent, so the uncertainty of a combination of both can be off.